# MIMIC-IV-Ext CDS Diagnosis Prototype

This notebook turns the CDS archive into a concrete diagnosis-task prototype. It loads the archive, builds compact per-case structures keyed by `stay_id`, constructs a simple nearest-case retrieval baseline, and shows the question format that can later feed the argumentation framework.

In [ ]:
from collections import Counter
from dataclasses import asdict, dataclass
import ast
import io
import math
from pathlib import Path
import re
import zipfile

import pandas as pd

pd.set_option("display.max_colwidth", 160)
pd.set_option("display.max_columns", 30)

In [ ]:
ZIP_PATH = Path("data/mimic-iv-ext-clinical-decision-support-for-referral-triage-and-diagnosis-1.0.2 (1).zip")
ZIP_PATH

In [ ]:
def read_csv_from_outer_zip(zip_path: Path, member_suffix: str) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path) as archive:
        member_name = next(name for name in archive.namelist() if name.endswith(member_suffix))
        with archive.open(member_name) as raw_file:
            return pd.read_csv(raw_file)


def read_csv_from_nested_zip(zip_path: Path, outer_member_suffix: str, inner_member_suffix: str) -> pd.DataFrame:
    with zipfile.ZipFile(zip_path) as archive:
        outer_member_name = next(name for name in archive.namelist() if name.endswith(outer_member_suffix))
        nested_bytes = archive.read(outer_member_name)
    with zipfile.ZipFile(io.BytesIO(nested_bytes)) as nested_archive:
        inner_member_name = next(name for name in nested_archive.namelist() if name.endswith(inner_member_suffix))
        with nested_archive.open(inner_member_name) as inner_file:
            return pd.read_csv(inner_file)

In [ ]:
clinical_data = read_csv_from_nested_zip(ZIP_PATH, "clinical_data.csv.zip", "clinical_data.csv")
diagnosis = read_csv_from_outer_zip(ZIP_PATH, "diagnosis.csv")
initial_assessment = read_csv_from_outer_zip(ZIP_PATH, "initial_assessment_info.csv")
patient_demographics = read_csv_from_outer_zip(ZIP_PATH, "patient_demographics.csv")
vital_signs = read_csv_from_outer_zip(ZIP_PATH, "vital_signs.csv")

In [ ]:
diagnosis.columns.tolist(), clinical_data.columns.tolist()

In [ ]:
def parse_list_column(value: object) -> list[str]:
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return []
    text = str(value).strip()
    if not text:
        return []
    try:
        parsed = ast.literal_eval(text)
    except (SyntaxError, ValueError):
        return [text]
    if isinstance(parsed, list):
        return [str(item) for item in parsed]
    return [str(parsed)]


TOKEN_RE = re.compile(r"[a-z0-9]+")
STOPWORDS = {
    "a", "an", "and", "are", "as", "at", "be", "by", "for", "from", "had", "has", "have", "he", "her",
    "his", "in", "is", "it", "of", "on", "or", "patient", "she", "that", "the", "their", "to", "was", "were", "with",
}


def tokenize(text: str) -> set[str]:
    return {
        token
        for token in TOKEN_RE.findall(text.lower())
        if len(token) > 2 and token not in STOPWORDS
    }


In [ ]:
@dataclass(frozen=True)
class DiagnosisCase:
    stay_id: int
    hpi: str
    patient_info: str
    initial_vitals: str
    tests: str
    past_medication: str
    chiefcomplaint: str
    triage: str
    primary_diagnosis: tuple[str, ...]
    secondary_diagnosis: tuple[str, ...]


@dataclass(frozen=True)
class RetrievedCase:
    stay_id: int
    score: float
    primary_diagnosis: tuple[str, ...]
    chiefcomplaint: str
    hpi_preview: str


In [ ]:
cases = diagnosis.merge(
    clinical_data[["stay_id", "tests", "past_medication", "text"]],
    on="stay_id",
    how="left",
).merge(
    initial_assessment[["stay_id", "triage", "chiefcomplaint"]],
    on="stay_id",
    how="left",
)

cases["primary_diagnosis_list"] = cases["primary_diagnosis"].map(parse_list_column)
cases["secondary_diagnosis_list"] = cases["secondary_diagnosis"].map(parse_list_column)

def build_case(row: pd.Series) -> DiagnosisCase:
    return DiagnosisCase(
        stay_id=int(row["stay_id"]),
        hpi=str(row["HPI"] or ""),
        patient_info=str(row["patient_info"] or ""),
        initial_vitals=str(row["initial_vitals"] or ""),
        tests=str(row.get("tests") or ""),
        past_medication=str(row.get("past_medication") or ""),
        chiefcomplaint=str(row.get("chiefcomplaint") or ""),
        triage=str(row.get("triage") or ""),
        primary_diagnosis=tuple(row["primary_diagnosis_list"]),
        secondary_diagnosis=tuple(row["secondary_diagnosis_list"]),
    )


case_objects = [build_case(row) for _, row in cases.iterrows()]
len(case_objects)

In [ ]:
label_counter = Counter()
for case in case_objects:
    label_counter.update(case.primary_diagnosis)

pd.DataFrame(label_counter.most_common(20), columns=["primary_diagnosis", "count"])

In [ ]:
def build_case_text(case: DiagnosisCase) -> str:
    parts = [
        case.hpi,
        case.chiefcomplaint,
        case.patient_info,
        case.initial_vitals,
        case.tests,
    ]
    return "\n".join(part for part in parts if part)


case_tokens = {case.stay_id: tokenize(build_case_text(case)) for case in case_objects}


def jaccard_similarity(left: set[str], right: set[str]) -> float:
    if not left or not right:
        return 0.0
    overlap = left & right
    union = left | right
    return len(overlap) / len(union) if union else 0.0


def nearest_cases(target_case: DiagnosisCase, top_k: int = 5) -> list[RetrievedCase]:
    target_tokens = case_tokens[target_case.stay_id]
    ranked: list[RetrievedCase] = []
    for candidate in case_objects:
        if candidate.stay_id == target_case.stay_id:
            continue
        score = jaccard_similarity(target_tokens, case_tokens[candidate.stay_id])
        if score <= 0:
            continue
        ranked.append(
            RetrievedCase(
                stay_id=candidate.stay_id,
                score=score,
                primary_diagnosis=candidate.primary_diagnosis,
                chiefcomplaint=candidate.chiefcomplaint,
                hpi_preview=" ".join(candidate.hpi.split())[:220],
            )
        )
    ranked.sort(key=lambda item: item.score, reverse=True)
    return ranked[:top_k]


## Diagnosis task prompt shape

This is the question form the repo can use for a diagnosis-task head over a shared case graph.

In [ ]:
def diagnosis_question(case: DiagnosisCase) -> str:
    return (
        "What primary diagnosis is best supported by this case?\n\n"
        f"HPI:\n{case.hpi.strip()}\n\n"
        f"Patient info:\n{case.patient_info.strip()}\n\n"
        f"Initial vitals:\n{case.initial_vitals.strip()}\n\n"
        f"Chief complaint:\n{case.chiefcomplaint.strip()}\n\n"
        f"Tests:\n{case.tests.strip()}\n"
    )


example_case = case_objects[0]
print(diagnosis_question(example_case)[:1800])

## Similar-case retrieval baseline

This is not the final GraphRAG. It is a notebook-only sanity check that the dataset supports case-based retrieval using patient-specific evidence.

In [ ]:
retrieved = nearest_cases(example_case, top_k=5)
pd.DataFrame([asdict(item) for item in retrieved])

In [ ]:
sample_indices = [0, 25, 100, 250, 500]
prototype_rows = []
for index in sample_indices:
    case = case_objects[index]
    top_matches = nearest_cases(case, top_k=3)
    prototype_rows.append(
        {
            "stay_id": case.stay_id,
            "gold_primary_diagnosis": list(case.primary_diagnosis),
            "chiefcomplaint": case.chiefcomplaint,
            "triage": case.triage,
            "top_match_labels": [list(item.primary_diagnosis) for item in top_matches],
            "top_match_scores": [round(item.score, 4) for item in top_matches],
        }
    )

pd.DataFrame(prototype_rows)

## Case graph shape

A repo rewrite can keep one shared case graph and then add task-specific evaluators for diagnosis, triage, and specialty referral.

In [ ]:
@dataclass(frozen=True)
class CaseNode:
    node_id: str
    label: str
    text: str


@dataclass(frozen=True)
class CaseEdge:
    source_id: str
    relation: str
    target_id: str


@dataclass(frozen=True)
class CaseGraph:
    stay_id: int
    nodes: tuple[CaseNode, ...]
    edges: tuple[CaseEdge, ...]
    gold_targets: dict[str, object]


def build_case_graph(case: DiagnosisCase) -> CaseGraph:
    nodes = (
        CaseNode(f"case:{case.stay_id}", "CASE", f"stay_id={case.stay_id}"),
        CaseNode(f"case:{case.stay_id}:hpi", "HPI", case.hpi),
        CaseNode(f"case:{case.stay_id}:patient_info", "PATIENT_INFO", case.patient_info),
        CaseNode(f"case:{case.stay_id}:vitals", "INITIAL_VITALS", case.initial_vitals),
        CaseNode(f"case:{case.stay_id}:chiefcomplaint", "CHIEF_COMPLAINT", case.chiefcomplaint),
        CaseNode(f"case:{case.stay_id}:tests", "TESTS", case.tests),
        CaseNode(f"case:{case.stay_id}:medications", "PAST_MEDICATION", case.past_medication),
    )
    edges = (
        CaseEdge(f"case:{case.stay_id}", "HAS_HPI", f"case:{case.stay_id}:hpi"),
        CaseEdge(f"case:{case.stay_id}", "HAS_PATIENT_INFO", f"case:{case.stay_id}:patient_info"),
        CaseEdge(f"case:{case.stay_id}", "HAS_INITIAL_VITALS", f"case:{case.stay_id}:vitals"),
        CaseEdge(f"case:{case.stay_id}", "HAS_CHIEF_COMPLAINT", f"case:{case.stay_id}:chiefcomplaint"),
        CaseEdge(f"case:{case.stay_id}", "HAS_TESTS", f"case:{case.stay_id}:tests"),
        CaseEdge(f"case:{case.stay_id}", "HAS_PAST_MEDICATION", f"case:{case.stay_id}:medications"),
    )
    gold_targets = {
        "diagnosis": list(case.primary_diagnosis),
        "secondary_diagnosis": list(case.secondary_diagnosis),
        "triage": case.triage,
    }
    return CaseGraph(stay_id=case.stay_id, nodes=nodes, edges=edges, gold_targets=gold_targets)


graph = build_case_graph(example_case)
len(graph.nodes), len(graph.edges), graph.gold_targets

In [ ]:
pd.DataFrame([asdict(node) for node in graph.nodes])

## Readout

If this prototype is acceptable, the next repo step is not a full rewrite in one shot. The safer path is:

1. build a clean CDS ingestion module
2. materialize one shared case graph artifact keyed by `stay_id`
3. implement a diagnosis-only evaluator first
4. add triage and specialty referral as separate task heads over the same graph
